In [1]:
from FCN_model import FCN_tune
import copy
import json
import random

import numpy as np
import optuna
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)


c:\Users\96561\miniconda3\envs\UCR-pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
MAX_EPOCHS = 100
PATIENCE = 15


In [3]:


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(42)
        torch.cuda.manual_seed_all(42)
        torch.backends.cudnn.deterministic = True


set_seed(SEED)



In [4]:
FordA_train = np.load("FordA_processed/FordA_TRAIN.npy")
FordA_test = np.load("FordA_processed/FordA_TEST.npy")

In [5]:
FordA_X_train = np.expand_dims(FordA_train[:, 1:], axis=1)
FordA_y_train = FordA_train[:, 0]
FordA_y_train = (FordA_y_train == 1).astype(np.int64)

In [6]:
FordA_X_test = np.expand_dims(FordA_test[:, 1:], axis=1)
FordA_y_test = FordA_test[:, 0]
FordA_y_test = (FordA_y_test == 1).astype(np.int64)

In [7]:
FordA_X_train = torch.tensor(FordA_X_train, dtype=torch.float32)
FordA_y_train = torch.tensor(FordA_y_train, dtype=torch.long)

In [8]:
FordA_X_test = torch.tensor(FordA_X_test, dtype=torch.float32)
FordA_y_test = torch.tensor(FordA_y_test, dtype=torch.long)

In [9]:

# ---------------------------------------------------
# 1. Prepare training/validation datasets
# ---------------------------------------------------

train_idx, val_idx = train_test_split(
    np.arange(len(FordA_y_train)),
    test_size=0.1,
    stratify=FordA_y_train.numpy(),
    random_state=SEED
)

train_ds = TensorDataset(FordA_X_train[train_idx], FordA_y_train[train_idx])
val_ds = TensorDataset(FordA_X_train[val_idx], FordA_y_train[val_idx])
test_ds = TensorDataset(FordA_X_test, FordA_y_test)

# train_loader = DataLoader(
#     train_ds,
#     batch_size=360, # calculated as FordA_X_train.shape[0] / 10, according to tensorflow version
#     shuffle=True
# )

# val_loader = DataLoader(
#     val_ds,
#     batch_size=360, # calculated as FordA_X_train.shape[0] / 10, according to tensorflow version
#     shuffle=True
# )

# val_loader = DataLoader(
#     val_ds,
#     batch_size=360, # calculated as FordA_X_train.shape[0] / 10, according to tensorflow version
#     shuffle=True
# )


In [ ]:

# ---------------------------------------------------
# 2. Define training and validation functions
# ---------------------------------------------------

def train_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to("cuda")
        y_batch = y_batch.to("cuda")

        optimizer.zero_grad(set_to_none=True)

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y_batch)
        correct += (logits.argmax(1) == y_batch).sum().item()
        total += len(y_batch)

    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total = 0

    all_predictions = []
    all_labels = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to("cuda")
        y_batch = y_batch.to("cuda")

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        predictions = logits.argmax(dim=1)

        total_loss += loss.item() * len(y_batch)
        total += len(y_batch)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(y_batch.cpu().tolist())

    accuracy = accuracy_score(
        all_labels, all_predictions
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            all_labels,
            all_predictions,
            average="macro",
            labels=[0, 1],
            zero_division=0
        )
    )

    return {
        "loss": total_loss / total,
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1)
    }



# ---------------------------------------------------
# 3. Define the Optuna objective
# ---------------------------------------------------

def objective(trial):
    set_seed(SEED)

    filters = (
        trial.suggest_categorical("filters1", [64, 128, 256]),
        trial.suggest_categorical("filters2", [128, 256, 512]),
        trial.suggest_categorical("filters3", [64, 128, 256])

    )

    kernels = (
        trial.suggest_categorical("kernel1", [8, 16, 6]),
        trial.suggest_categorical("kernel2", [5, 10, 4]),
        trial.suggest_categorical("kernel3", [3, 6, 2])
    )

    dropouts = (
            trial.suggest_categorical("dropout1", [0.2, 0.5, 0.7]),
            trial.suggest_categorical("dropout2", [0.2, 0.5, 0.7]),
            trial.suggest_categorical("dropout3", [0.2, 0.5, 0.7])
        )

    lr = trial.suggest_float(
        "learning_rate", 1e-5, 1e-2, log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size", [16, 32, 64, 128, 256]
    )

    weight_decay = trial.suggest_categorical(
        "weight_decay",
        [0.0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2]
    )

    optimizer_name = trial.suggest_categorical(
        "optimizer", ["Adam", "AdamW"]
    )

    # train_loader, val_loader = make_loaders(batch_size)
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=256,
        shuffle=False
    )

    # Replace with your existing FCN constructor.
    origin_FCN = FCN_tune(
        filters=filters,
        kernels=kernels,
        dropouts=dropouts
    ).to("cuda")
    

    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(
            origin_FCN.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )
    else:
        optimizer = torch.optim.AdamW(
            origin_FCN.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )

    criterion = nn.CrossEntropyLoss()

    best_f1 = -1.0
    best_loss = float("inf")
    best_epoch = 0
    best_state = None
    best_metrics = None
    patience_counter = 0

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
        "val_precision": [],
        "val_recall": [],
        "val_f1": []
    }

    for epoch in range(MAX_EPOCHS):

        train_loss, train_acc = train_epoch(
            origin_FCN,
            train_loader,
            optimizer,
            criterion
        )

        metrics = validate(
            origin_FCN,
            val_loader,
            criterion
        )

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_acc)
        history["val_loss"].append(metrics["loss"])
        history["val_accuracy"].append(metrics["accuracy"])
        history["val_precision"].append(metrics["precision"])
        history["val_recall"].append(metrics["recall"])
        history["val_f1"].append(metrics["f1"])

        # Select best checkpoint by macro F1,
        # using validation loss as a tie-breaker.
        improved = (
            metrics["f1"] > best_f1
            or (
                metrics["f1"] == best_f1
                and metrics["loss"] < best_loss
            )
        )

        if improved:
            best_f1 = metrics["f1"]
            best_loss = metrics["loss"]
            best_epoch = epoch + 1

            best_metrics = metrics.copy()

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in origin_FCN.state_dict().items()
            }

            patience_counter = 0
        else:
            patience_counter += 1

        # Report current macro F1 to Optuna.
        trial.report(metrics["f1"], step=epoch)

        if trial.should_prune():
            raise optuna.TrialPruned()

        if patience_counter >= PATIENCE:
            break

    # Save metrics from the selected best epoch
    for name, value in best_metrics.items():
        trial.set_user_attr(
            f"best_{name}", float(value)
        )

    trial.set_user_attr("best_epoch", best_epoch)

    torch.save(
        {
            "model_state_dict": best_state,
            "params": trial.params,
            "best_epoch": best_epoch,
            "best_metrics": best_metrics,
            "history": history
        },
        f"Tuned_model/fcn_trial_{trial.number}.pt"
    )

    return best_f1



# ---------------------------------------------------
# 4. Run optimisation
# ---------------------------------------------------

study = optuna.create_study(
    study_name="FordA_FCN_MacroF1",
    storage="sqlite:///fcn_optuna.db",
    load_if_exists=True,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=50,
        interval_steps=10
    )
)

study.enqueue_trial({
    "filters1": 128,
    "filters2": 256,
    "filters3": 128,
    "kernel1": 8,
    "kernel2": 5,
    "kernel3": 3,
    "learning_rate": 0.001,
    "batch_size": 128,
    "weight_decay": 0.0,
    "optimizer": "Adam"
}, skip_if_exists=True)

study.optimize(
    objective,
    n_trials=50,
    gc_after_trial=True
)

best = study.best_trial

print(f"Best trial: {best.number}")
print(f"Best epoch: {best.user_attrs['best_epoch']}")

for metric in ["accuracy", "precision", "recall", "f1"]:
    print(
        f"{metric}: "
        f"{best.user_attrs[f'best_{metric}']:.4f}"
    )

print("Best parameters:", study.best_params)

[I 2026-10-17 23:16:56,506] Using an existing study with `study_name='FordA_FCN_MacroF1'` instead of creating a new one.
[I 2026-10-17 23:16:56,517] Trial with params {'filters1': 128, 'filters2': 256, 'filters3': 128, 'kernel1': 8, 'kernel2': 5, 'kernel3': 3, 'learning_rate': 0.001, 'batch_size': 128, 'weight_decay': 0.0, 'optimizer': 'Adam'} already exists. Skipping enqueue.
c:\Users\96561\miniconda3\envs\UCR-pytorch\Lib\site-packages\torch\nn\modules\conv.py:380: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1105.)
  return F.conv1d(
[I 2026-10-17 23:17:15,561] Trial 6 finished with value: 0.9224227911831522 and parameters: {'filters1': 128, 'filters2': 128, 'filters3': 128, 'kernel1': 6, 'kernel2': 5, 'kernel3': 2, 'learning_rate': 0.00019762189340280086, 'batch_size': 32, 'weight_decay': 1e-06, 'o

Best trial: 25
Best epoch: 23
accuracy: 0.9584
precision: 0.9584
recall: 0.9585
f1: 0.9584
Best parameters: {'filters1': 256, 'filters2': 128, 'filters3': 64, 'kernel1': 8, 'kernel2': 10, 'kernel3': 6, 'learning_rate': 0.004038721483970501, 'batch_size': 64, 'weight_decay': 0.0, 'optimizer': 'AdamW'}


In [11]:
import pandas as pd

rows = []

for trial in study.trials:
    if trial.state != optuna.trial.TrialState.COMPLETE:
        continue

    rows.append({
        "trial": trial.number,
        "accuracy": trial.user_attrs["best_accuracy"],
        "precision": trial.user_attrs["best_precision"],
        "recall": trial.user_attrs["best_recall"],
        "f1": trial.user_attrs["best_f1"],
        "epoch": trial.user_attrs["best_epoch"],
        **trial.params
    })

results = pd.DataFrame(rows)
results = results.sort_values("f1", ascending=False)

print(results.head(10))

results.to_csv("optuna_metrics.csv", index=False)

    trial  accuracy  precision    recall        f1  epoch  filters1  filters2  \
20     25  0.958449   0.958379  0.958492  0.958428     23       256       128   
18     23  0.955679   0.955648  0.955928  0.955670     36       256       128   
15     20  0.955679   0.955648  0.955928  0.955670     44       256       128   
21     26  0.955679   0.955648  0.955928  0.955670     55       256       256   
12     17  0.955679   0.955651  0.955651  0.955651     42       256       512   
34     39  0.952909   0.953946  0.953639  0.952907     44       256       256   
38     43  0.952909   0.952947  0.953225  0.952903     45       256       128   
17     22  0.952909   0.952837  0.952948  0.952885     39        64       128   
40     45  0.950139   0.950550  0.950660  0.950138     22       256       128   
48     53  0.950139   0.950550  0.950660  0.950138     33       256       512   

    filters3  kernel1  kernel2  kernel3  learning_rate  batch_size  \
20        64        8       10        

In [12]:

# Load best trial's checkpoint
checkpoint = torch.load(
    f"fcn_trial_{study.best_trial.number}.pt",
    map_location="cpu",
    weights_only=True
)

# Retrieve architecture hyperparameters
params = checkpoint["params"]

filters = (
    params["filters1"],
    params["filters2"],
    params["filters3"]
)

kernels = (
    params["kernel1"],
    params["kernel2"],
    params["kernel3"]
)

# Reconstruct the selected FCN architecture
best_model = FCN_tune(
    in_channels=1,
    num_classes=2,
    filters=filters,
    kernels=kernels
).to("cuda")

# Restore trained weights
best_model.load_state_dict(
    checkpoint["model_state_dict"]
)

best_model.eval()

print("Loaded trial:", study.best_trial.number)
print("Architecture filters:", filters)
print("Architecture kernels:", kernels)
print("Selected epoch:", checkpoint["best_epoch"])
print("Validation metrics:", checkpoint["best_metrics"])


FileNotFoundError: [Errno 2] No such file or directory: 'fcn_trial_25.pt'

In [ ]:
test_loader = DataLoader(
        test_ds,
        batch_size=256,
        shuffle=False
    )
criterion = nn.CrossEntropyLoss()

test_metrics = validate(
    best_model,
    test_loader,
    criterion
)

for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")